In [ ]:
!pip install beautifulsoup4 pandas lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 29.5 MB/s  0:00:00 eta 0:00:01


In [ ]:
!pip install thefuzz python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 2.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [python-Levenshtein]


In [11]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from thefuzz import fuzz
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.internationalcrimesdatabase.org"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

LETTERS = [
    "A","B","C","Č","D","Đ","E","F","G","H","I","J","K","L",
    "M","N","O","P","R","S","Š","T","U","V","W","X","Y","Z"
]

# ── Session with automatic retries ──────────────────────────────────────────
def make_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=5,                        # retry up to 5 times
        backoff_factor=2,               # wait 2, 4, 8, 16... seconds between retries
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update(HEADERS)
    return session

SESSION = make_session()

def safe_get(url: str, retries: int = 5, backoff: float = 3.0):
    """GET with manual retry on ConnectionReset errors the adapter won't catch."""
    for attempt in range(retries):
        try:
            resp = SESSION.get(url, timeout=30)
            resp.raise_for_status()
            return resp
        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError) as e:
            wait = backoff * (2 ** attempt)
            print(f"  Connection error on attempt {attempt+1}/{retries}: {e}")
            print(f"  Waiting {wait:.0f}s before retry...")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {url} after {retries} attempts")

def is_initials_only(name: str) -> bool:
    tokens = re.split(r'[\s,\-]+', name.strip())
    tokens = [t for t in tokens if t]
    if not tokens:
        return False
    return all(re.fullmatch(r'[A-Za-z]{1,2}\.?', t) for t in tokens)

def all_name_variants(row) -> list[str]:
    variants = [row["case_name"]]
    if row["other_names"] != "N/A":
        variants += [n.strip() for n in row["other_names"].split(" | ")]
    return [v for v in variants if v]

def fuzzy_deduplicate(df: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    kept_indices = []
    seen_variant_lists = []
    for i, row in df.iterrows():
        this_variants = all_name_variants(row)
        is_dup = False
        for seen_variants in seen_variant_lists:
            for a in this_variants:
                for b in seen_variants:
                    if fuzz.token_sort_ratio(a, b) >= threshold:
                        is_dup = True
                        break
                if is_dup:
                    break
            if is_dup:
                break
        if not is_dup:
            kept_indices.append(i)
            seen_variant_lists.append(this_variants)
    removed = len(df) - len(kept_indices)
    print(f"Fuzzy dedup removed {removed} near-duplicates ({len(kept_indices)} remain)")
    return df.loc[kept_indices].reset_index(drop=True)

def get_all_case_links() -> list[str]:
    links = []
    seen_ids = set()
    for letter in LETTERS:
        resp = safe_get(f"{BASE_URL}/Cases/ByName/{letter}")
        soup = BeautifulSoup(resp.text, "lxml")
        found = 0
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if not href.startswith("/Case/"):
                continue
            match = re.match(r'/Case/(\d+)/', href)
            if match:
                case_id = match.group(1)
                if case_id not in seen_ids:
                    seen_ids.add(case_id)
                    links.append(BASE_URL + href)
                    found += 1
      
    return links

def classify_court_type(court: str) -> str:
    """Classify court into: Domestic courts, Hybrid tribunals, UN ad-hoc tribunals, Regional HR bodies"""
    if court == "N/A":
        return "N/A"
    
    court_lower = court.lower()
    
    # UN ad-hoc tribunals
    un_ad_hoc_patterns = [
        r'international criminal tribunal for (?:rwanda|the former yugoslavia)',
        r'ictr|icty',
        r'(?:special tribunal|tribunal spécial) for lebanon',
        r'stl-',
        r'extraordinary chambers.*cambodia',
        r'eccc',
    ]
    for pattern in un_ad_hoc_patterns:
        if re.search(pattern, court_lower):
            return "UN ad-hoc tribunals"
    
    # Hybrid tribunals (mixed international/domestic)
    hybrid_patterns = [
        r'special panels?.*serious crimes',
        r'court of (?:bosnia|cambodia).*war crimes',
        r'international crimes tribunal.*bangladesh',
        r'iraqi high tribunal',
        r'khmer rouge|dili|east timor',
        r'residual mechanism',
    ]
    for pattern in hybrid_patterns:
        if re.search(pattern, court_lower):
            return "Hybrid tribunals"
    
    # Regional HR bodies (UN, African, Inter-American, European)
    regional_patterns = [
        r'international court of justice',
        r'icj',
        r'permanent court',
        r'european court of human rights',
        r'echr',
        r'inter-?american court',
        r'african court',
        r'african commission',
        r'un human rights',
        r'united nations',
        r'commission on human rights',
    ]
    for pattern in regional_patterns:
        if re.search(pattern, court_lower):
            return "Regional HR bodies"
    
    # Default to domestic courts (national courts of any country)
    return "Domestic courts"

def parse_case(url: str) -> dict:
    resp = safe_get(url)
    soup = BeautifulSoup(resp.text, "lxml")

    h1 = soup.find("h1")
    case_name = h1.get_text(strip=True) if h1 else "N/A"

    court, parties, crimes, case_number, decision_date, other_names = (
        "N/A", "N/A", "N/A", "N/A", "N/A", "N/A"
    )
    table = soup.find("table")
    if table:
        for row in table.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) < 2:
                continue
            label = cells[0].get_text(strip=True).lower()
            value = cells[1].get_text(" | ", strip=True)
            if "court" in label:
                court = value
            elif "case number" in label:
                case_number = value
            elif "decision date" in label:
                decision_date = value
            elif "other name" in label:
                other_names = value
            elif "parties" in label:
                raw_parties = [p.strip() for p in value.split(" | ")]
                named = [p for p in raw_parties if not is_initials_only(p)]
                parties = " | ".join(named) if named else "N/A"
            elif "categor" in label:
                crimes = value

    if court != "N/A" and "," in court:
        country = court.rsplit(",", 1)[-1].strip()
    else:
        country = court

    court_type = classify_court_type(court)


    return {
        "case_name":     case_name,
        "other_names":   other_names,
        "case_number":   case_number,
        "decision_date": decision_date,
        "court":         court,
        "country":       country,
        "crimes":        crimes,
        "court_type":    court_type,
        "parties":       parties,
        "url":           url,
    }

def scrape_icd(output_csv="icd_cases.csv", delay=1.5, limit=None):
    links = get_all_case_links()
    if limit:
        links = links[:limit]

    records = []
    for i, url in enumerate(links, 1):
        print(f"[{i}/{len(links)}] {url}")
        try:
            records.append(parse_case(url))
        except Exception as e:
            print(f"  ERROR: {e}")
        time.sleep(delay)

    df = pd.DataFrame(records)

    before = len(df)
    has_number = df[df["case_number"] != "N/A"].drop_duplicates(subset="case_number", keep="first")
    no_number  = df[df["case_number"] == "N/A"]
    df = pd.concat([has_number, no_number]).reset_index(drop=True)
    print(f"Case number dedup removed {before - len(df)} duplicates")

    df = fuzzy_deduplicate(df, threshold=90)

    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} cases to {output_csv}")
    return df

df = scrape_icd()


df.head()

[1/731] https://www.internationalcrimesdatabase.org/Case/3264/A-v-Secretary-of-State-for-the-Home-Department-(No-1)/
[2/731] https://www.internationalcrimesdatabase.org/Case/901/A-v-Secretary-of-State-for-the-Home-Department-(No-2)/
[3/731] https://www.internationalcrimesdatabase.org/Case/831/A-(Khaled-Nezzar)/
[4/731] https://www.internationalcrimesdatabase.org/Case/1002/A-A-Z-et-al-v-Franks-et-al/
[5/731] https://www.internationalcrimesdatabase.org/Case/902/A-and-B-v-State-of-Israel/
[6/731] https://www.internationalcrimesdatabase.org/Case/1071/A-Mukantagara/
[7/731] https://www.internationalcrimesdatabase.org/Case/925/A-v-The-Minister-of-Defence/
[8/731] https://www.internationalcrimesdatabase.org/Case/201/Abdah-et-al/
[9/731] https://www.internationalcrimesdatabase.org/Case/200/Abdah-et-al/
[10/731] https://www.internationalcrimesdatabase.org/Case/876/Abdulmutallab/
[11/731] https://www.internationalcrimesdatabase.org/Case/903/Abebe-Jira-v-Negewo/
[12/731] https://www.international

,case_name,other_names,case_number,decision_date,court,country,crimes,court_type,parties,url
0,A (FC) and others (FC) (Appellants) v. Secreta...,A v. Secretary of State for the Home Departmen...,[2004] UKHL 56,16 December 2004,"House of Lords, Great Britain (UK)",Great Britain (UK),Terrorism,Domestic courts,Mahmoud Abu Rideh | Jamal Ajouaou | Secretary ...,https://www.internationalcrimesdatabase.org/Ca...
1,"A v. Ministère Public de la Confédération, B a...",N/A,BB.2011.140,25 July 2012,"Federal Criminal Court, Switzerland",Switzerland,"Torture, War crimes",Domestic courts,Ministère Public de la Confédération/Public Mi...,https://www.internationalcrimesdatabase.org/Ca...
2,A. A. Z. et al. v. Tommy Franks et al.,N/A,JC041E2_2,14 January 2004,"Cour de Cassation, Section Francaise, 2e Chamb...",Belgium,War crimes,Domestic courts,"A.-A.R. M. | A.T. | A.R. A. | A.N., K., M. | R...",https://www.internationalcrimesdatabase.org/Ca...
3,A. and B. v. State of Israel,N/A,"CrimA 6659/06, CrimA 1757/07, CrimA 8228/07, C...",11 June 2008,The Supreme Court of Israel sitting as the Cou...,Israel,War crimes,Domestic courts,State of Israel,https://www.internationalcrimesdatabase.org/Ca...
4,Public Prosecutor v. A. Mukantagara et al./ Le...,N/A,RPA 3/Gc/R1/RUH,30 June 1998,Court of Appeal of Ruhengeri / Cour d'Appel de...,Rwanda,"Crimes against humanity, Genocide, War crimes",Domestic courts,Agnès Mukantagara | Jean Damascène Nzayisenga ...,https://www.internationalcrimesdatabase.org/Ca...


In [8]:
import pandas as pd
import plotly.express as px
  

df = pd.read_csv('icd_cases.csv')

counts = df['court_type'].value_counts().reset_index()
counts.columns = ['court_type', 'count']

fig = px.pie(
    counts,
    values='count',
    names='court_type',
    hole=0.4
)

fig.write_image('pie.pdf')

/var/folders/pr/vjys5__s3fx2c0txz6_pd1880000gn/T/ipykernel_1426/1462516146.py:17: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [10]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('icd_cases.csv')

df['year'] = df['decision_date'].str.extract(r'(\d{4})').astype(float)

df['crimes'] = df['crimes'].str.split(', ')
df = df.explode('crimes')

df = df.dropna(subset=['year', 'crimes', 'country'])
df = df[df['crimes'] != 'N/A']

top5 = df['country'].value_counts().nlargest(5).index
df = df[df['country'].isin(top5)]

fig = px.scatter(
    df,
    x='year',
    y='crimes',
    color='country',
)
fig.update_traces(marker=dict(size=10))
fig.update_layout(
    height=700,
    yaxis={'categoryorder': 'total ascending'},
    legend=dict(title='Country', orientation='v')
)

fig.write_image('dot.pdf', width=1600, height=900, scale=2)

/var/folders/pr/vjys5__s3fx2c0txz6_pd1880000gn/T/ipykernel_4311/331842642.py:30: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('icd_cases.csv')

top10 = df['country'].value_counts().nlargest(10).index

df_top = df[df['country'].isin(top10)].copy()

summary = df_top.groupby('country').apply(lambda x: pd.Series({
    'total': len(x),
    'genocide': x['crimes'].str.contains('Genocide', na=False).sum()
})).reset_index()

summary['other'] = summary['total'] - summary['genocide']

# Order by total cases descending (matching the image)
summary = summary.sort_values('genocide', ascending=False)

summary = summary.sort_values('genocide', ascending=False)
summary = summary[summary['genocide'] > 1]

melted = summary.melt(id_vars='country', value_vars=['other', 'genocide'],
                      var_name='type', value_name='count')

fig = px.bar(
    melted,
    x='country',
    y='count',
    color='type',
    color_discrete_map={'genocide': '#e24b4a', 'other': '#b5d4f4'},
    category_orders={
        'country': summary['country'].tolist(),
        'type': ['other', 'genocide']
    },
    labels={'count': 'Number of cases', 'country': 'Country', 'type': ''},
    title='Cases by country — genocide vs. other (top 10 countries)',
)

fig.update_traces(marker_line_width=0)

fig.update_layout(
    height=550,
    xaxis=dict(tickangle=-30, gridcolor='rgba(255,255,255,0.05)'),
    yaxis=dict(title='Number of cases', gridcolor='rgba(255,255,255,0.1)'), 
    legend=dict(title='Country', orientation='v')
)

fig.write_image('genocide.pdf', width=1600, height=900, scale=2)

/var/folders/pr/vjys5__s3fx2c0txz6_pd1880000gn/T/ipykernel_4311/2757774017.py:10: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/pr/vjys5__s3fx2c0txz6_pd1880000gn/T/ipykernel_4311/2757774017.py:49: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).




In [8]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    'Status': ['Rome Statute Parties', 'Non-Parties (incl. US, Israel)'],
    'Cases': [216, 139]
})

fig = px.bar(df, y='Status', x='Cases', orientation='h',
             color='Status', color_discrete_sequence=['#2c5aa0', '#a63d40'])

# Remove redundant legend and axis title
fig.update_layout(
    showlegend=False,
    yaxis_title=None,
    width=700,
    height=250,
    margin=dict(l=20, r=20, t=20, b=20)
)

fig.write_image('rome.pdf')


/var/folders/pr/vjys5__s3fx2c0txz6_pd1880000gn/T/ipykernel_864/2728143551.py:21: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


